# Topic: Secure Chat Simulator - Hybrid encryption, RSA public/private keys exchange, and ephemeral AES session chats
 
## 1. HYBRID CRYPTOGRAPHY
- **The Problem**:
  - Asymmetric encryption (RSA) is mathematically complex and slow, making it unsuitable for encrypting large volumes of real-time stream data (like chat messages).
  - Symmetric encryption (AES) is extremely fast, but requires sharing a secret key beforehand. If the key is intercepted during exchange, the channel is compromised.
- **The Solution (Hybrid Encryption)**:
  1. **Key Exchange (Asymmetric)**: Use RSA to securely exchange a one-time random **Symmetric Session Key** (AES key). The sender encrypts the session key with the receiver's RSA public key.
  2. **Data Transfer (Symmetric)**: Once both parties have the session key, all subsequent messages are encrypted using fast symmetric AES-GCM.
 
---


In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# 1. Simulating Key Generation for Alice and Bob
print("--- Generating RSA Key Pairs ---")
# Alice's Keys
alice_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
alice_public_key = alice_private_key.public_key()

# Bob's Keys
bob_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
bob_public_key = bob_private_key.public_key()

print("RSA Key Pairs successfully generated.")



In [ ]:
# 2. Key Exchange: Bob wants to start a secure chat with Alice
# Bob generates an ephemeral symmetric session key (AES-256)
ephemeral_session_key = AESGCM.generate_key(bit_length=256)
print(f"\nBob's generated session key (raw): {ephemeral_session_key.hex()}")

# Bob encrypts (wraps) the session key using Alice's RSA Public Key
encrypted_session_key = alice_public_key.encrypt(
    ephemeral_session_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print(f"Encrypted session key size: {len(encrypted_session_key)} bytes")

# Bob transmits this encrypted key over the network to Alice.
# Alice decrypts the session key using her RSA Private Key
alice_decrypted_session_key = alice_private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print(f"Alice's decrypted session key:      {alice_decrypted_session_key.hex()}")
print(f"Do session keys match?              {ephemeral_session_key == alice_decrypted_session_key}")



In [ ]:
# 3. Secure Symmetric Messaging (AES-GCM)
# Both parties now use the session key to chat
alice_aes = AESGCM(alice_decrypted_session_key)
bob_aes = AESGCM(ephemeral_session_key)

# Bob sends encrypted message to Alice
message_to_alice = b"Hello Alice, this is Bob. Meet me at 10:00."
iv_1 = os.urandom(12)
encrypted_msg = bob_aes.encrypt(iv_1, message_to_alice, None)

print(f"\n--- Encrypted Payload Transmitted ---")
print(f"IV (hex):      {iv_1.hex()}")
print(f"Cipher (hex):  {encrypted_msg.hex()}")

# Alice receives and decrypts message
decrypted_msg = alice_aes.decrypt(iv_1, encrypted_msg, None)
print(f"\nAlice Decrypted Output: {decrypted_msg.decode('utf-8')}")
